In [0]:
spark.sql("USE CATALOG irctc_catalog")
spark.sql("USE SCHEMA bronze")

In [0]:
df_exp = spark.read \
    .option("multiline", "true") \
    .json("/Volumes/irctc_catalog/landing/raw_files/EXP-TRAINS.json")

In [0]:
display(df_exp)

In [0]:
df_exp.printSchema()

In [0]:
from pyspark.sql.functions import *

df_exp = df_exp \
    .withColumn("source_file", lit("EXP-TRAINS.json")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("train_category", lit("EXP"))


In [0]:
display(df_exp)

In [0]:
from pyspark.sql.functions import *

df_pass = (
    spark.read.option("multiline", "true")
    .json("/Volumes/irctc_catalog/landing/raw_files/PASS-TRAINS.json")
    .withColumn("source_file", lit("PASS-TRAINS.json"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("train_category", lit("PASS"))
)

In [0]:
from pyspark.sql.functions import *

df_sf = (
    spark.read.option("multiline", "true")
    .json("/Volumes/irctc_catalog/landing/raw_files/SF-TRAINS.json")
    .withColumn("source_file", lit("SF-TRAINS.json"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("train_category", lit("SF"))
)


In [0]:
df_all = df_exp \
    .unionByName(df_pass) \
    .unionByName(df_sf)

In [0]:
df_all.count()
display(df_all)

In [0]:
df_all.groupBy("train_category").count().show()

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS irctc_catalog.bronze
""")

In [0]:
spark.sql("SHOW SCHEMAS IN irctc_catalog").display()

In [0]:
df_all.write.format("delta").mode("overwrite").saveAsTable("irctc_catalog.bronze.train_raw")

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM irctc_catalog.bronze.train_raw
""").show()